# YgoMaster-PackBuilder — Pack Assembly Tests

Notebook offline per testare:

```text
ManifestPack
+
YgoProDeckPackData
+
PackType
+
Pack rules
+
rarity mapping
+
YDK mapping
        ↓
assemble_pack_definition(...)
        ↓
PackDefinition
```

Obiettivi:

- verificare l'assemblaggio di main pack, mini-pack e shared staples;
- controllare i valori derivati dalle pack rules;
- verificare l'inheritance della opening policy;
- verificare gli override;
- controllare la conversione `ManifestPackSource -> SourcePackRef`;
- verificare normalizzazione rarità e risoluzione Master Duel ID;
- verificare il comportamento con carte YDK irrisolte;
- verificare il controllo sull'`external_id`.

Questo notebook non effettua richieste di rete.


## 1. Ambiente e root repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

print("Repository root:", repo_root)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.assembly import (
    assemble_pack_definition,
)

from ygomaster_packbuilder.domain import (
    PackDefinition,
    PackType,
    Rarity,
    SourceProvider,
)

from ygomaster_packbuilder.manifest.models import (
    ManifestPack,
    ManifestPackSource,
    ManifestSharedPack,
)

from ygomaster_packbuilder.rules import get_pack_rule

from ygomaster_packbuilder.sources.ygoprodeck import (
    YgoProDeckPackCard,
    YgoProDeckPackData,
)

print("Imports OK")

Imports OK


## 3. Mapping di test

In [3]:
rarity_mapping = {
    "Common": Rarity.N,
    "Rare": Rarity.R,
    "Super Rare": Rarity.SR,
    "Ultra Rare": Rarity.UR,
}

ydk_mapping = {
    1001: 501,
    1002: 502,
    1003: 503,
    1004: 504,
}

print("Rarity mapping:", rarity_mapping)
print("YDK mapping:", ydk_mapping)

Rarity mapping: {'Common': <Rarity.N: 'N'>, 'Rare': <Rarity.R: 'R'>, 'Super Rare': <Rarity.SR: 'SR'>, 'Ultra Rare': <Rarity.UR: 'UR'>}
YDK mapping: {1001: 501, 1002: 502, 1003: 503, 1004: 504}


## 4. Source pack di test

In [4]:
source_pack = YgoProDeckPackData(
    external_id="24116",
    name="Scaled Invasion",
    description="Offline assembly test pack",
    creator="livius03",
    image_id="8400623",
    cards=[
        YgoProDeckPackCard(
            name="Card N",
            ydk_id=1001,
            card_type="Effect Monster",
            source_rarity="Common",
        ),
        YgoProDeckPackCard(
            name="Card R",
            ydk_id=1002,
            card_type="Spell Card",
            source_rarity="Rare",
        ),
        YgoProDeckPackCard(
            name="Card SR",
            ydk_id=1003,
            card_type="Trap Card",
            source_rarity="Super Rare",
        ),
        YgoProDeckPackCard(
            name="Card UR",
            ydk_id=1004,
            card_type="Effect Monster",
            source_rarity="Ultra Rare",
        ),
    ],
)

print(source_pack)

YgoProDeckPackData(external_id='24116', name='Scaled Invasion', description='Offline assembly test pack', creator='livius03', image_id='8400623', cards=[YgoProDeckPackCard(name='Card N', ydk_id=1001, card_type='Effect Monster', source_rarity='Common'), YgoProDeckPackCard(name='Card R', ydk_id=1002, card_type='Spell Card', source_rarity='Rare'), YgoProDeckPackCard(name='Card SR', ydk_id=1003, card_type='Trap Card', source_rarity='Super Rare'), YgoProDeckPackCard(name='Card UR', ydk_id=1004, card_type='Effect Monster', source_rarity='Ultra Rare')])


## 5. Manifest main pack

In [5]:
manifest_main = ManifestPack(
    key="livio-scaled-invasion",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

print(manifest_main)

ManifestPack(key='livio-scaled-invasion', source=ManifestPackSource(provider='ygoprodeck', external_id='24116'), opening_policy=None)


## 6. Assemblaggio main pack

In [6]:
main_pack = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert isinstance(main_pack, PackDefinition)

print(main_pack)

PackDefinition(key='livio-scaled-invasion', name='Scaled Invasion', description='Offline assembly test pack', type=<PackType.PERSONAL_MAIN: 'personal_main'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=100, booster_size=7, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key=None, cards=[Card(name='Card N', ydk_id=1001, source_rarity='Common', master_duel_id=501, rarity=<Rarity.N: 'N'>), Card(name='Card R', ydk_id=1002, source_rarity='Rare', master_duel_id=502, rarity=<Rarity.R: 'R'>), Card(name='Card SR', ydk_id=1003, source_rarity='Super Rare', master_duel_id=503, rarity=<Rarity.SR: 'SR'>), Card(name='Card UR', ydk_id=1004, source_rarity='Ultra Rare', master_duel_id=504, rarity=<Rarity.UR: 'UR'>)])


## 7. Metadata e relazioni del main pack

In [7]:
assert main_pack.key == "livio-scaled-invasion"
assert main_pack.name == "Scaled Invasion"
assert main_pack.description == "Offline assembly test pack"

assert main_pack.type is PackType.PERSONAL_MAIN

assert main_pack.owner_key == "livio"
assert main_pack.parent_pack_key is None

assert main_pack.opening_policy_key == "configuration-a"

print("Main pack metadata OK")

Main pack metadata OK


## 8. Source reference domain

In [8]:
assert main_pack.source.provider is SourceProvider.YGOPRODECK
assert main_pack.source.external_id == "24116"

print(main_pack.source)

SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116')


## 9. Regole derivate del main pack

In [9]:
rule = get_pack_rule(PackType.PERSONAL_MAIN)

assert main_pack.expected_card_count == rule.expected_card_count
assert main_pack.booster_size == rule.booster_size

assert main_pack.expected_card_count == 100
assert main_pack.booster_size == 7

print("Main pack rule values OK")

Main pack rule values OK


## 10. Carte convertite e normalizzate

In [10]:
assert len(main_pack.cards) == 4

expected_rarities = [
    Rarity.N,
    Rarity.R,
    Rarity.SR,
    Rarity.UR,
]

assert [card.rarity for card in main_pack.cards] == expected_rarities

for card in main_pack.cards:
    print(
        card.name,
        "| source:", card.source_rarity,
        "| rarity:", card.rarity,
        "| MD:", card.master_duel_id,
    )

Card N | source: Common | rarity: N | MD: 501
Card R | source: Rare | rarity: R | MD: 502
Card SR | source: Super Rare | rarity: SR | MD: 503
Card UR | source: Ultra Rare | rarity: UR | MD: 504


## 11. Master Duel IDs risolti

In [11]:
expected_ids = [501, 502, 503, 504]

assert [
    card.master_duel_id
    for card in main_pack.cards
] == expected_ids

print("Master Duel ID resolution OK")

Master Duel ID resolution OK


## 12. Opening policy override

In [12]:
manifest_override = deepcopy(manifest_main)
manifest_override.opening_policy = "configuration-b"

pack = assemble_pack_definition(
    manifest_pack=manifest_override,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert pack.opening_policy_key == "configuration-b"

print("Explicit opening policy override OK")

Explicit opening policy override OK


## 13. Mini-pack

In [13]:
manifest_mini = ManifestPack(
    key="livio-mini-test",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

mini_pack = assemble_pack_definition(
    manifest_pack=manifest_mini,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MINI,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
    parent_pack_key="livio-scaled-invasion",
)

assert mini_pack.type is PackType.PERSONAL_MINI
assert mini_pack.owner_key == "livio"
assert mini_pack.parent_pack_key == "livio-scaled-invasion"

assert mini_pack.expected_card_count == 20
assert mini_pack.booster_size == 3

print(mini_pack)

PackDefinition(key='livio-mini-test', name='Scaled Invasion', description='Offline assembly test pack', type=<PackType.PERSONAL_MINI: 'personal_mini'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=20, booster_size=3, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key='livio-scaled-invasion', cards=[Card(name='Card N', ydk_id=1001, source_rarity='Common', master_duel_id=501, rarity=<Rarity.N: 'N'>), Card(name='Card R', ydk_id=1002, source_rarity='Rare', master_duel_id=502, rarity=<Rarity.R: 'R'>), Card(name='Card SR', ydk_id=1003, source_rarity='Super Rare', master_duel_id=503, rarity=<Rarity.SR: 'SR'>), Card(name='Card UR', ydk_id=1004, source_rarity='Ultra Rare', master_duel_id=504, rarity=<Rarity.UR: 'UR'>)])


## 14. Shared staples pack

In [14]:
manifest_shared = ManifestSharedPack(
    key="shared-staples",
    type="shared_staples",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

shared_pack = assemble_pack_definition(
    manifest_pack=manifest_shared,
    source_pack=source_pack,
    pack_type=PackType.SHARED_STAPLES,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
)

assert shared_pack.type is PackType.SHARED_STAPLES
assert shared_pack.owner_key is None
assert shared_pack.parent_pack_key is None

assert shared_pack.expected_card_count == 200
assert shared_pack.booster_size == 11

print(shared_pack)

PackDefinition(key='shared-staples', name='Scaled Invasion', description='Offline assembly test pack', type=<PackType.SHARED_STAPLES: 'shared_staples'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=200, booster_size=11, opening_policy_key='configuration-a', owner_key=None, parent_pack_key=None, cards=[Card(name='Card N', ydk_id=1001, source_rarity='Common', master_duel_id=501, rarity=<Rarity.N: 'N'>), Card(name='Card R', ydk_id=1002, source_rarity='Rare', master_duel_id=502, rarity=<Rarity.R: 'R'>), Card(name='Card SR', ydk_id=1003, source_rarity='Super Rare', master_duel_id=503, rarity=<Rarity.SR: 'SR'>), Card(name='Card UR', ydk_id=1004, source_rarity='Ultra Rare', master_duel_id=504, rarity=<Rarity.UR: 'UR'>)])


## 15. Carta YDK irrisolta

L'assembler non deve fallire solo perché una carta non è presente nel mapping YDK.
La carta deve restare con:

```text
master_duel_id = None
```

La domain validation la segnalerà più avanti.


In [15]:
source_with_missing_id = deepcopy(source_pack)

source_with_missing_id.cards.append(
    YgoProDeckPackCard(
        name="Unresolved Card",
        ydk_id=99999999,
        card_type="Effect Monster",
        source_rarity="Common",
    )
)

pack = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_with_missing_id,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

unresolved = [
    card
    for card in pack.cards
    if card.master_duel_id is None
]

assert len(unresolved) == 1
assert unresolved[0].ydk_id == 99999999
assert unresolved[0].rarity is Rarity.N

print("Unresolved card preserved:", unresolved[0])

Unresolved card preserved: Card(name='Unresolved Card', ydk_id=99999999, source_rarity='Common', master_duel_id=None, rarity=<Rarity.N: 'N'>)


## 16. Source external ID non corrispondente

In [16]:
wrong_source = deepcopy(source_pack)
wrong_source.external_id = "99999"

try:
    assemble_pack_definition(
        manifest_pack=manifest_main,
        source_pack=wrong_source,
        pack_type=PackType.PERSONAL_MAIN,
        default_opening_policy_key="configuration-a",
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        owner_key="livio",
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "L'assembler avrebbe dovuto rilevare external_id incoerente."
    )

Expected ValueError: Source pack external ID does not match manifest pack source external ID.


## 17. Rarità sorgente non mappata

L'assembler usa il normalization layer, quindi una rarità non supportata
deve ancora produrre errore.


In [17]:
source_with_unknown_rarity = deepcopy(source_pack)
source_with_unknown_rarity.cards[0].source_rarity = "Secret Rare"

try:
    assemble_pack_definition(
        manifest_pack=manifest_main,
        source_pack=source_with_unknown_rarity,
        pack_type=PackType.PERSONAL_MAIN,
        default_opening_policy_key="configuration-a",
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        owner_key="livio",
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "Secret Rare non mappata avrebbe dovuto produrre errore."
    )

Expected ValueError: Unsupported source rarity: 'Secret Rare'.


## 18. DTO sorgente non modificati

L'assembler deve produrre nuovi oggetti domain senza modificare le carte
contenute nel DTO YGOPRODeck.


In [18]:
source_copy = deepcopy(source_pack)

assembled = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_copy,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert source_copy.cards[0].source_rarity == "Common"
assert not hasattr(source_copy.cards[0], "rarity")
assert not hasattr(source_copy.cards[0], "master_duel_id")

assert assembled.cards[0] is not source_copy.cards[0]

print("Source DTO remains unchanged")

Source DTO remains unchanged
